# 🗂️ Notebook 2: Flash Sale — Data Model & APIs


## 🛠️ Setup

```bash
cd 06-system-designs/flash-sale
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window:
`Cmd+Shift+P` → **Reload Window**.

This lab uses **only pure Python** (plus `pydantic`) — no Docker, no external services.
We simulate Redis, queues, and threads in-process so every cell runs anywhere.


## Two storage layers

During a flash sale, the **hot path** reads and writes **Redis only**.
Primary databases (Postgres/MySQL) are too slow for 100k+ writes/sec on one row.

- **Redis (hot, during sale)**: stock counters, per-user caps, reservations with TTL.
- **Postgres (cold, after sale)**: durable orders, shipments, payments ledger.

A background worker drains reservations from Redis into Postgres as they get paid.


## 🔑 Redis key design

Picking good keys is 80% of making this work. Bad keys = hot-key problem (one Redis shard
melts while the rest are idle).

| Key | Type | Purpose | Example |
|---|---|---|---|
| `stock:{item_id}` | integer counter | remaining units | `stock:item-42 → 1000` |
| `user_cap:{item_id}:{user_id}` | int with TTL | per-user cap | `user_cap:item-42:u7 → 1` |
| `reservation:{reservation_id}` | hash + TTL | pending order | `{user, item, qty, status}` |
| `idem:{idempotency_key}` | string + TTL | retry-safe API | `idem:abc123 → reservation_id` |
| `queue:item-42` | stream / list | admission queue | consumed by workers |

### 🔥 Hot key mitigation

All traffic for one SKU hits one key. If that SKU is too hot for one Redis shard, split it:

```
stock:item-42:shard0   stock:item-42:shard1   ...   stock:item-42:shard15
```

Each request picks a shard (e.g., `hash(user_id) % 16`). Each shard holds `1000 / 16` units.
You trade a bit of "some shards sell out first" unfairness for **16x throughput**.
We'll see this trick again in notebook 3.


## 🌐 HTTP API

Three endpoints are enough:

```http
POST /flash/reserve
  Headers:  Idempotency-Key: <client-generated uuid>
  Body:     { "item_id": "item-42", "user_id": 7, "qty": 1 }
  200:      { "status": "reserved", "reservation_id": "...", "pay_by": "2026-04-20T12:10:00Z" }
  409:      { "status": "sold_out" }
  429:      { "status": "rate_limited", "retry_after_ms": 500 }
  503:      { "status": "queue_full" }

POST /flash/pay
  Body:     { "reservation_id": "...", "payment_token": "tok_xyz" }
  200:      { "status": "paid", "order_id": "..." }
  410:      { "status": "expired" }   # reservation TTL hit before user paid

GET  /flash/status/{reservation_id}
  200:      { "status": "pending|paid|expired|cancelled" }
```

Key API design rules for flash sales:

1. **Idempotency-Key on `/reserve`.** Clients will retry on timeouts. Without this, a retry
   looks like a second reservation → user gets a 409 they don't understand, *or* worse,
   you decrement stock twice.
2. **Short, structured responses.** `{"status": "sold_out"}` is parseable. A 500-byte HTML
   error page is not.
3. **`pay_by` deadline is visible to the client.** The frontend can show a countdown.


## 🧱 Pydantic models

Pydantic gives us free request validation and JSON schema. Exactly what FastAPI uses.


In [1]:
from datetime import datetime, timezone, timedelta
from enum import Enum
from pydantic import BaseModel, Field, field_validator


class ReserveRequest(BaseModel):
    item_id: str = Field(pattern=r"^item-[a-z0-9-]+$")
    user_id: int = Field(ge=1)
    qty: int = Field(default=1, ge=1, le=5)  # hard per-request cap


class ReservationStatus(str, Enum):
    pending = "pending"
    paid = "paid"
    expired = "expired"
    cancelled = "cancelled"


class ReservationResponse(BaseModel):
    status: ReservationStatus
    reservation_id: str
    pay_by: datetime

    @field_validator("pay_by")
    @classmethod
    def must_be_future(cls, v: datetime) -> datetime:
        if v <= datetime.now(timezone.utc):
            raise ValueError("pay_by must be in the future")
        return v


# Happy-path example
req = ReserveRequest(item_id="item-42", user_id=7, qty=1)
print("Request :", req.model_dump_json())

resp = ReservationResponse(
    status=ReservationStatus.pending,
    reservation_id="r-abc123",
    pay_by=datetime.now(timezone.utc) + timedelta(minutes=10),
)
print("Response:", resp.model_dump_json())


Request : {"item_id":"item-42","user_id":7,"qty":1}
Response: {"status":"pending","reservation_id":"r-abc123","pay_by":"2026-04-19T22:14:44.369945Z"}


### Validation catches bad input *before* the hot path

Bad requests should die at the edge — never touch Redis.
Below we show pydantic rejecting malformed input. This is CPU we save on the critical path.


In [2]:
from pydantic import ValidationError

bad_cases = [
    {"item_id": "item-42", "user_id": 0, "qty": 1},             # user_id must be >= 1
    {"item_id": "../etc/passwd", "user_id": 7, "qty": 1},       # item_id pattern
    {"item_id": "item-42", "user_id": 7, "qty": 99},            # qty too large
]
for case in bad_cases:
    try:
        ReserveRequest(**case)
    except ValidationError as e:
        # Print first error only for compactness
        err = e.errors()[0]
        print(f"✗ rejected {case}: {err['loc']} — {err['msg']}")


✗ rejected {'item_id': 'item-42', 'user_id': 0, 'qty': 1}: ('user_id',) — Input should be greater than or equal to 1
✗ rejected {'item_id': '../etc/passwd', 'user_id': 7, 'qty': 1}: ('item_id',) — String should match pattern '^item-[a-z0-9-]+$'
✗ rejected {'item_id': 'item-42', 'user_id': 7, 'qty': 99}: ('qty',) — Input should be less than or equal to 5


## 🪪 Idempotency in practice

When the client retries, the server must return the *same* answer, not create a second
reservation. The pattern:

1. Client generates a random `Idempotency-Key` (UUID).
2. Server stores `idem:{key} → reservation_id` with TTL (say 10 minutes).
3. If the same key comes back → return the cached reservation instead of decrementing stock.


In [3]:
import uuid
from typing import Optional


class FakeRedis:
    """Tiny in-memory stand-in so we can demo idempotency without a server."""
    def __init__(self): self.kv: dict[str, str] = {}
    def setnx(self, k: str, v: str) -> bool:
        if k in self.kv: return False
        self.kv[k] = v
        return True
    def get(self, k: str) -> Optional[str]: return self.kv.get(k)


redis = FakeRedis()
stock = {"item-42": 10}


def reserve(item_id: str, user_id: int, idem_key: str) -> dict:
    cached = redis.get(f"idem:{idem_key}")
    if cached is not None:
        return {"status": "reserved", "reservation_id": cached, "cached": True}

    if stock[item_id] <= 0:
        return {"status": "sold_out"}

    stock[item_id] -= 1
    rid = f"r-{uuid.uuid4().hex[:8]}"
    redis.setnx(f"idem:{idem_key}", rid)
    return {"status": "reserved", "reservation_id": rid, "cached": False}


key = "client-generated-uuid-1"
print("First call :", reserve("item-42", 7, key))
print("Retry (same key):", reserve("item-42", 7, key))  # same result, no extra decrement
print("Stock remaining :", stock["item-42"])            # decremented exactly once
assert stock["item-42"] == 9, "idempotent reserve must decrement only once"
print("✓ retries are safe")


First call : {'status': 'reserved', 'reservation_id': 'r-cf4a070d', 'cached': False}
Retry (same key): {'status': 'reserved', 'reservation_id': 'r-cf4a070d', 'cached': True}
Stock remaining : 9
✓ retries are safe


## 🧾 Postgres schema (post-sale durable store)

After the sale is over, a worker moves paid reservations into a proper relational store.

```sql
CREATE TABLE orders (
    order_id        UUID PRIMARY KEY,
    user_id         BIGINT      NOT NULL,
    item_id         TEXT        NOT NULL,
    qty             INT         NOT NULL,
    reserved_at     TIMESTAMPTZ NOT NULL,
    paid_at         TIMESTAMPTZ NOT NULL,
    amount_cents    BIGINT      NOT NULL,
    payment_ref     TEXT        NOT NULL UNIQUE
);
CREATE INDEX ON orders (user_id);
CREATE INDEX ON orders (item_id, paid_at);
```

Notes:

- `payment_ref UNIQUE` is the final guard against double-charging.
- We do **not** put `stock` in Postgres. During the sale, Postgres would be the bottleneck.
- After the sale, you can reconcile: `SELECT count(*) FROM orders WHERE item_id='item-42';`
  should equal `initial_stock - redis_stock_remaining`.
